# Language-Model Adapter — DIMER artifact inference tutorial (standalone)

[![GitHub](https://img.shields.io/badge/GitHub-181717?style=flat&logo=github&logoColor=white)](https://github.com/kurtvalcorza/language-model-pipeline) [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kurtvalcorza/language-model-pipeline/blob/main/tutorials/language_model_artifact_inference_colab.ipynb) [![Hugging Face](https://img.shields.io/badge/%F0%9F%A4%97%20Hugging%20Face-HuggingFaceTB%2FSmolLM2--360M--Instruct-ffcc4d?style=flat)](https://huggingface.co/HuggingFaceTB/SmolLM2-360M-Instruct) [![Upstream](https://img.shields.io/badge/Upstream-huggingface%2Fsmollm-181717?style=flat&logo=github&logoColor=white)](https://github.com/huggingface/smollm) [![arXiv](https://img.shields.io/badge/arXiv-2502.02737-b31b1b.svg)](https://arxiv.org/abs/2502.02737)

**Profile:** `ARTIFACT-INFERENCE`  
**Mode:** `GUIDED`  
**Notebook specification:** DIMER Notebook Specification 2.0 — **standalone** (§4)  
**Capability:** consume an externally supplied PEFT adapter bundle, verify its manifest and provenance against the pinned `HuggingFaceTB/SmolLM2-360M-Instruct` base snapshot, attach it, and generate text for new prompts

**This notebook is standalone.** It carries the repository's pipeline module (`src/lmpipeline/pipeline.py` at revision `9240c5226e92`) verbatim in Section 2, the pinned model identity and the per-file SHA-256 manifest in Section 3, and the exact runtime pins in Section 1, so it keeps working after export even if the repository changes or disappears. Its only external dependencies are the pinned Python distributions and the Hugging Face Hub at the immutable revision `a10cc1512eabd3dde888204e902eca88bddb4951` (~727 MB, digest-verified before loading). It was generated by `tools/build_notebook.py` (build_notebook.py/2); edit the repository and regenerate rather than editing cells.

**Run all:** **Known NOTEBOOK_SPEC 2.0 gap (§19, SART1/RUN2):** the default path does not yet obtain a trusted sample adapter bundle automatically — with `ARTIFACT_DIR` empty, Section 4 opens an upload dialog for a bundle produced by the E2E tutorial; an executor sets `ARTIFACT_DIR` to a directory already in the runtime to skip the dialog. Until a published sample bundle is wired in, this notebook is a `Candidate`, not release-grade. Once the bundle is present, **Run all** installs the pinned dependencies, stages and digest-verifies the pinned base snapshot, verifies the bundle manifest before any state is deserialised, attaches the adapter to the verified base, validates new prompts into an input manifest, generates with the adapter off and on and with sampling, writes the evaluation report, and exports outputs and provenance — all inside this kernel, with no DIMER worker or service and no credential.

**Bring Your Own Data:** New-input BYOD is the `CUSTOM_PROMPT` form field in Section 6 (empty by default): your own prompt passes through the same validation, generation and export cells as the sample prompts. A user-supplied adapter bundle is the separate optional `ARTIFACT_DIR`/upload branch in Section 4, verified by `verify_artifact_bundle` before deserialisation. Uploads stay inside this runtime; do not upload confidential or restricted data unless you are authorised to process it here.

Someone hands you an adapter ZIP and says "this is the fine-tuned model". It is not a model; it is a **PEFT adapter**: a few million low-rank delta weights that only mean something on top of one specific base model at one specific revision. This notebook consumes a bundle produced **outside this execution** (for example by the E2E tutorial in a separate session), treats it as untrusted input, verifies every file against the bundle's manifest and its provenance against the carried module's pinned identity, attaches it to the digest-verified base snapshot, and generates with the adapter switched off and on. No training happens and **no artifact is created here**. The carried module supplies `extract_zip_safely`, `verify_artifact_bundle`, `LanguageModelPipeline.load_adapter`, `validate_prompts` and `evaluation_report`.

**Trust boundary.** Manifest and digest checks establish that the bundle is internally consistent and names this pipeline's base model; they do not authenticate the sender. The bundle format carries no pickle and no Python: adapter weights are `safetensors`, the tokenizer and configs are JSON/text, `trust_remote_code` is never enabled, and the base weights are acquired separately in Section 3 and digest-verified before use. Use only bundles from a producer you trust, and paste the whole-archive SHA-256 they gave you into `EXPECTED_ARTIFACT_ZIP_SHA256`.

**Learning objectives:** install the pinned runtime, read what the carried pipeline module guarantees, resolve and digest-verify the immutable base snapshot, supply an externally produced adapter bundle and verify it — archive path safety, per-file digests, format, provenance, pinned base identity — before any model state is deserialised, inspect its provenance, attach the adapter to the verified base with no network fallback, validate new prompts into an input manifest, generate with the adapter off and on under an explicit greedy decoding rule and then with sampling, produce an evaluation report that is `not-measurable` because no labelled data exists, and export machine-readable results plus provenance.

**This notebook does not demonstrate:** artifact creation, fine-tuning of any kind, merging the adapter into the base weights, serving, quality claims of any kind (without labelled prompts nothing is measured), and any base model other than the pinned `smollm2-360m` snapshot — a bundle trained on another base or revision is refused, not adapted.

## Prerequisites

- **Runtime:** a fresh supported runtime (Google Colab or Jupyter, Python 3.12). A CUDA device is recommended (the base loads in 4-bit `nf4`, as it was trained); on CPU the module loads the base in float32, which is slower but works for a few prompts. The pinned `torch==2.14.0` install is the largest download of the run.
- **Artifact:** an externally produced adapter bundle — the E2E tutorial writes `outputs/language_model_finetuning_adapter_bundle.zip` — supplied through the upload dialog, or as a directory already present in the runtime via `ARTIFACT_DIR` for non-interactive execution. Nothing in this notebook manufactures it.
- **Data:** new prompts typed into the form (two Filipino prompts are prefilled). No dataset is bundled, because scoring self-generated rows would not be external-artifact evidence. Do not upload confidential or restricted data to a hosted notebook environment unless you are authorized to do so. Uploaded inputs remain in the notebook runtime; this pipeline does not send them to a third-party inference API.
- **External access:** the Hugging Face Hub only, to fetch the pinned `HuggingFaceTB/SmolLM2-360M-Instruct` snapshot (~727 MB in total) at revision `a10cc1512eab…`. No GitHub access and no credentials are required; nothing is installed from this repository.

## 1. Install the pinned runtime

The dependency set is pinned exactly (the same pins as the repository's pyproject.toml at the generating revision; any `--index-url`/`--find-links` lines are passed to pip as written) and installed directly — there is no repository clone and no package install. If a pin replaces a distribution this runtime has already imported, the cell stops with a restart instruction rather than continuing with mixed versions. Look for a dictionary reporting the notebook's source revision, Python, `torch`, `transformers` versions, and whether CUDA is available.

In [ ]:
import importlib
import importlib.metadata
import os
import platform
import subprocess
import sys

PINS = [
    'PyYAML==6.0.3',
    'torch==2.14.0',
    'transformers==4.57.1',
    'tokenizers==0.22.1',
    'huggingface-hub==0.36.2',
    'peft==0.18.0',
    'accelerate==1.11.0',
    'bitsandbytes==0.49.0',
    'safetensors==0.8.0',
    'datasets==5.0.1',
    'pandas==3.0.5',
]
NOTEBOOK_SOURCE = {
    'repository': 'language-model-pipeline',
    'repository_revision': '9240c5226e927d5a11ff63625bb909b8603e412b',
    'embedded_module': 'src/lmpipeline/pipeline.py',
    'embedded_modules': ['src/lmpipeline/pipeline.py'],
    'module_sha256': '9605aa15ba46279db513e749dfca04a39956d223ae55e433acf2f01d82a9af78',
    'generator': 'build_notebook.py/2',
    'notebook_spec': '2.0',
}
SKIP_INSTALL = os.environ.get('DIMER_NOTEBOOK_CI_PREINSTALLED') == '1'

def _installed_version(distribution):
    try:
        return importlib.metadata.version(distribution)
    except importlib.metadata.PackageNotFoundError:
        return None

if not SKIP_INSTALL:
    # Capture every distribution already imported in this runtime, whatever its module name
    # (PIL -> pillow), so a pinned install that replaces a loaded package is detected and the
    # notebook stops with a restart instruction instead of continuing with mixed versions.
    _module_dists = importlib.metadata.packages_distributions()
    _loaded = sorted({d for m in list(sys.modules) for d in _module_dists.get(m.partition('.')[0], ())})
    loaded = {distribution: _installed_version(distribution) for distribution in _loaded}
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *PINS], check=True)
    importlib.invalidate_caches()
    stale = []
    for distribution, before in loaded.items():
        installed = _installed_version(distribution)
        if before is not None and before != installed:
            stale.append(f'{distribution}: loaded={before}, installed={installed}')
    if stale:
        raise RuntimeError('Core dependencies changed while older modules were loaded: ' + '; '.join(stale) + '. Restart the runtime, then rerun from the top.')

import torch, transformers
print({'notebook_source': NOTEBOOK_SOURCE, 'python': platform.python_version(), 'torch': torch.__version__, 'transformers': transformers.__version__, 'cuda': torch.cuda.is_available()})

## 2. Pipeline code (carried verbatim from `src/lmpipeline/` @ `9240c5226e92`)

The next 1 cell(s) **are** the repository's package, module by module in dependency order: the pinned identity constants, snapshot verification (`verify_snapshot`), staged download (`stage_missing_files`), the named operational ceilings, the public validation and evaluation helpers, and the pipeline class. The text is the modules', byte for byte, except for the rewrite rules listed in `tools/build_notebook.py` (1 rule(s), plus the removal of package-relative `from .x import` lines, whose names are already defined by the preceding cells). The repository's parity test (`tests/test_notebook_parity.py`) fails whenever these cells and the modules diverge, so what you run here is what the repository tests. Nothing in these cells runs a model yet.

**Module 1/1:** `src/lmpipeline/pipeline.py`

In [ ]:
"""Standalone tutorial pipeline for the DIMER language-model capability.

This module is what the standalone tutorials carry verbatim (NOTEBOOK_SPEC 1.1 §3.6): the
pinned base-model identity, manifest-driven snapshot verification and staging, chat rendering,
assistant-only loss masking, greedy generation, the PEFT adapter-bundle contract, and the public
``validate_inputs`` / ``evaluation_report`` stage helpers. The QLoRA training loop itself stays
in the tutorial (it is deliberately plain PyTorch so every step is visible); this module owns
everything the loop consumes and everything the exported artifact must satisfy.

Heavy libraries (``torch``, ``transformers``, ``peft``) are imported lazily inside the functions
that need them so the contract helpers stay importable and testable offline.
"""

from __future__ import annotations

import hashlib
import json
import math
import shutil
import stat
import zipfile
from collections.abc import Callable, Mapping, Sequence
from dataclasses import dataclass, field
from pathlib import Path, PurePosixPath
from typing import Any

MODEL_ID = "HuggingFaceTB/SmolLM2-360M-Instruct"
MODEL_REVISION = "a10cc1512eabd3dde888204e902eca88bddb4951"
MODEL_LICENSE = "apache-2.0"
MODEL_KEY = "smollm2-360m"
DEFAULT_WEIGHTS_DIR = Path.cwd() / "weights" / MODEL_KEY  # standalone rewrite (build_notebook.py): working-directory-relative
MANIFEST_NAME = "dimer-base-manifest.json"
WEIGHTS_FILE = "model.safetensors"

# Registry ceilings for the pinned model (src/lmpipeline/data/model-registry.yaml, smollm2-360m).
MAX_SEQUENCE_LENGTH_CEILING = 2048  # tokens; the registry's hard ceiling for this base model
DEFAULT_MAX_SEQUENCE_LENGTH = 512  # tokens; the tutorial default, well inside the ceiling
MAX_TOTAL_TRAIN_TOKENS = 50_000_000  # tokens across the training split per run
MIN_TRAIN_EXAMPLES = 2  # one for training and one for the manufactured validation split
IGNORE_INDEX = -100  # PyTorch cross-entropy ignore_index: unsupervised label positions
ALLOWED_ROLES = frozenset({"system", "user", "assistant"})
ARTIFACT_FORMAT = "peft_adapter"
ARTIFACT_FORMAT_VERSION = 1
ARTIFACT_MANIFEST_NAME = "artifact-manifest.json"
DECODING_RULE = "greedy"  # do_sample=False: deterministic given weights, device and versions
MAX_NEW_TOKENS_CEILING = 1024  # generated tokens per call; the tutorials use 16..96
MAX_PROMPT_CHARS = 20_000  # characters per prompt turn; rendered prompts must fit the ceiling

# The other base models the repository's DIMER worker admits. Informational in the standalone
# tutorial: the notebook pins MODEL_ID/MODEL_REVISION above and carries that snapshot's manifest;
# switching base model means regenerating the notebook from a template that pins another key.
TUTORIAL_REGISTRY: dict[str, dict[str, Any]] = {
    "smollm2-360m": {"model_id": MODEL_ID, "revision": MODEL_REVISION, "license": "apache-2.0", "min_vram_gb": None, "requires_hf_token": False, "dimer_zip": True, "state": "tutorial-default/apache-2.0"},  # noqa: E501
    "qwen3-0.6b": {"model_id": "Qwen/Qwen3-0.6B", "revision": "c1899de289a04d12100db370d81485cdf75e47ca", "license": "apache-2.0", "min_vram_gb": None, "requires_hf_token": False, "dimer_zip": True, "state": "smoke-ci/apache-2.0"},  # noqa: E501
    "smollm3-3b": {"model_id": "HuggingFaceTB/SmolLM3-3B", "revision": "a07cc9a04f16550a088caea529712d1d335b0ac1", "license": "apache-2.0", "min_vram_gb": None, "requires_hf_token": False, "dimer_zip": True, "state": "tutorial-candidate/internal-only"},  # noqa: E501
    "qwen3-1.7b": {"model_id": "Qwen/Qwen3-1.7B", "revision": "70d244cc86ccca08cf5af4e1e306ecf908b1ad5e", "license": "apache-2.0", "min_vram_gb": 9.3, "requires_hf_token": False, "dimer_zip": True, "state": "user-facing"},  # noqa: E501
    "qwen3-4b": {"model_id": "Qwen/Qwen3-4B", "revision": "1cfa9a7208912126459214e8b04321603b3df60c", "license": "apache-2.0", "min_vram_gb": 11.5, "requires_hf_token": False, "dimer_zip": True, "state": "user-facing"},  # noqa: E501
    "granite-4.1-3b": {"model_id": "ibm-granite/granite-4.1-3b", "revision": "c0650403e44e78ec0262dab1c90914c65b196c4e", "license": "apache-2.0", "min_vram_gb": 8.7, "requires_hf_token": False, "dimer_zip": True, "state": "user-facing"},  # noqa: E501
    "deepseek-r1-distill-qwen-1.5b": {"model_id": "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B", "revision": "ad9f0ae0864d7fbcd1cd905e3c6c5b069cc8b562", "license": "mit", "min_vram_gb": None, "requires_hf_token": False, "dimer_zip": True, "state": "open-reasoning/mit"},  # noqa: E501
    "qwen2.5-coder-1.5b": {"model_id": "Qwen/Qwen2.5-Coder-1.5B-Instruct", "revision": "2e1fd397ee46e1388853d2af2c993145b0f1098a", "license": "apache-2.0", "min_vram_gb": None, "requires_hf_token": False, "dimer_zip": True, "state": "open-code/apache-2.0"},  # noqa: E501
    "smollm2-1.7b": {"model_id": "HuggingFaceTB/SmolLM2-1.7B-Instruct", "revision": "31b70e2e869a7173562077fd711b654946d38674", "license": "apache-2.0", "min_vram_gb": None, "requires_hf_token": False, "dimer_zip": True, "state": "open-small/apache-2.0"},  # noqa: E501
    "granite-3.1-2b-instruct": {"model_id": "ibm-granite/granite-3.1-2b-instruct", "revision": "bbc2aed595bd38bd770263dc3ab831db9794441d", "license": "apache-2.0", "min_vram_gb": None, "requires_hf_token": False, "dimer_zip": True, "state": "enterprise-transparent/apache-2.0"},  # noqa: E501
    "h2o-danube3-4b-chat": {"model_id": "h2oai/h2o-danube3-4b-chat", "revision": "1e5c6fa6620f8bf078958069ab4581cd88e0202c", "license": "apache-2.0", "min_vram_gb": None, "requires_hf_token": False, "dimer_zip": True, "state": "mobile-edge/apache-2.0"},  # noqa: E501
    "llama-3.2-3b-instruct": {"model_id": "meta-llama/Llama-3.2-3B-Instruct", "revision": "0cb88a4f764b7a12671c53f0838cd831a0843b95", "license": "llama3.2", "min_vram_gb": None, "requires_hf_token": True, "dimer_zip": False, "state": "credential-test-candidate"},  # noqa: E501
}

# Pinned public samples the tutorial can load (id, immutable dataset revision, license).
SAMPLE_DATASETS: dict[str, dict[str, str]] = {
    "Sample: Filipino SFT": {
        "dataset_id": "jpaulpoliquit/ph-sft-ai-authored-v1",
        "revision": "8333699c6cc7296cc69cefc09def010851ded919",
        "license": "apache-2.0",
    },
    "Sample: Dolly": {
        "dataset_id": "databricks/databricks-dolly-15k",
        "revision": "bdd27f4d94b9c1f951818a7da7fd7aeea5dbff1a",
        "license": "cc-by-sa-3.0",
    },
}


# ---------------------------------------------------------------------------
# Snapshot verification and staging (MOD6-MOD8, ST3/ST4)
# ---------------------------------------------------------------------------


def sha256_of_file(path: str | Path) -> str:
    digest = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()


def verify_snapshot(path: str | Path | None = None) -> dict[str, Any]:
    """Check a local base-model snapshot against its manifest; raise naming the first mismatch."""
    root = Path(path or DEFAULT_WEIGHTS_DIR)
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        raise FileNotFoundError(f"snapshot manifest not found: {manifest_path}")
    with open(manifest_path, encoding="utf-8") as fh:
        manifest = json.load(fh)
    if manifest.get("format") != "dimer_hf_snapshot" or manifest.get("formatVersion") != 1:
        raise ValueError("unsupported snapshot manifest format (expected dimer_hf_snapshot/1)")
    if manifest.get("modelId") != MODEL_ID:
        raise ValueError(f"manifest modelId {manifest.get('modelId')!r} != {MODEL_ID!r}")
    if manifest.get("revision") != MODEL_REVISION:
        raise ValueError(f"manifest revision {manifest.get('revision')!r} != {MODEL_REVISION!r}")
    if not any(entry["path"].endswith(".safetensors") for entry in manifest.get("files", [])):
        raise ValueError("manifest lists no .safetensors weights")
    for entry in manifest.get("files", []):
        file_path = root / entry["path"]
        if not file_path.is_file():
            raise FileNotFoundError(f"snapshot file missing: {file_path}")
        size = file_path.stat().st_size
        if size != entry["bytes"]:
            raise ValueError(f"{entry['path']}: size {size} != manifest {entry['bytes']}")
        digest = sha256_of_file(file_path)
        if digest != entry["sha256"]:
            raise ValueError(f"{entry['path']}: sha256 {digest} != manifest {entry['sha256']}")
    return {"path": str(root), **manifest}


def _hub_download(relative_path: str, root: Path) -> None:
    """Fetch one manifest-listed file at MODEL_REVISION straight into the snapshot directory."""
    from huggingface_hub import hf_hub_download

    hf_hub_download(MODEL_ID, relative_path, revision=MODEL_REVISION, local_dir=str(root))


def stage_missing_files(
    path: str | Path | None = None,
    *,
    allow_download: bool = False,
    downloader: Callable[[str, Path], None] | None = None,
) -> list[str]:
    """Fetch manifest-listed files that are absent locally (a fresh clone commits the manifest but
    git-ignores the weights). Returns the relative paths fetched; ``verify_snapshot`` still runs
    afterwards."""
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        raise FileNotFoundError(f"manifest not found: {manifest_path}")
    with open(manifest_path, encoding="utf-8") as fh:
        manifest = json.load(fh)
    if manifest.get("modelId") != MODEL_ID or manifest.get("revision") != MODEL_REVISION:
        raise ValueError(
            f"manifest names {manifest.get('modelId')}@{manifest.get('revision')}, "
            f"package pins {MODEL_ID}@{MODEL_REVISION}; refusing to stage"
        )
    files = manifest["files"]
    missing = [entry["path"] for entry in files if not (root / entry["path"]).is_file()]
    if not missing:
        return []
    if not allow_download:
        raise FileNotFoundError(
            f"snapshot at {root} is missing {missing}; "
            f"pass allow_download=True to fetch them at {MODEL_REVISION}"
        )
    fetch = downloader or _hub_download
    for relative_path in missing:
        fetch(relative_path, root)
    return missing


# ---------------------------------------------------------------------------
# Dataset normalisation and hygiene (DAT*)
# ---------------------------------------------------------------------------


def canonical(record: Mapping[str, Any]) -> dict[str, Any]:
    """Normalize one source record into ``{"messages": [...]}``; raise if the schema is unknown."""
    if "messages" in record:
        messages = [{"role": m["role"], "content": str(m["content"])} for m in record["messages"]]
    elif "prompt" in record and ("completion" in record or "response" in record):
        answer = record.get("completion", record.get("response"))
        messages = [
            {"role": "user", "content": str(record["prompt"])},
            {"role": "assistant", "content": str(answer)},
        ]
    elif "instruction" in record and ("output" in record or "response" in record):
        context = record.get("input") or record.get("context")
        question = str(record["instruction"]) + (f"\n\n{context}" if context else "")
        answer = record.get("output", record.get("response"))
        messages = [
            {"role": "user", "content": question},
            {"role": "assistant", "content": str(answer)},
        ]
    else:
        raise ValueError(
            "Unsupported SFT schema: expected messages, prompt/completion, or instruction/output"
        )
    if any(m["role"] not in ALLOWED_ROLES for m in messages):
        raise ValueError("Invalid role in record")
    if not any(m["role"] == "assistant" and m["content"].strip() for m in messages):
        raise ValueError("Record has no non-empty assistant turn to learn from")
    return {"messages": messages}


def fingerprint(record: Mapping[str, Any]) -> str:
    """Stable identity for a record: SHA-256 of its canonical JSON."""
    payload = json.dumps(record, sort_keys=True, ensure_ascii=False, separators=(",", ":"))
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()


def dataset_digest(splits: Mapping[str, Sequence[Mapping[str, Any]]]) -> str:
    """Identity of an exact dataset: SHA-256 over every split's record fingerprints, in order."""
    joined = "".join(fingerprint(r) for name in sorted(splits) for r in splits[name])
    return hashlib.sha256(joined.encode("utf-8")).hexdigest()


def manufacture_validation(
    records: Sequence[Mapping[str, Any]], fraction: float = 0.2
) -> dict[str, list[dict[str, Any]]]:
    """Hold out the first ``fraction`` of ``records`` (already in a deterministic order) as
    validation. The split is manufactured from the training source, so its loss is an
    optimisation signal, not a task-quality measurement."""
    rows = [dict(r) for r in records]
    if len(rows) < MIN_TRAIN_EXAMPLES:
        raise ValueError(f"at least MIN_TRAIN_EXAMPLES={MIN_TRAIN_EXAMPLES} records are required")
    validation_size = max(1, int(len(rows) * fraction))
    return {"train": rows[validation_size:], "validation": rows[:validation_size]}


INPUT_SCHEMA: dict[str, Any] = {
    "input": "splits: {'train': [...], optional 'validation'/'test': [...]} of canonical chat "
    "records {'messages': [{'role', 'content'}, ...]} (messages, prompt/completion and "
    "instruction/output source schemas are normalised by `canonical`)",
    "roles": sorted(ALLOWED_ROLES),
    "assistant_turns": "every record needs at least one non-empty assistant turn",
    "train_examples": [MIN_TRAIN_EXAMPLES, None],
    "sequence_tokens": [1, MAX_SEQUENCE_LENGTH_CEILING],
    "max_sequence_length_default": DEFAULT_MAX_SEQUENCE_LENGTH,
    "train_tokens_total": [1, MAX_TOTAL_TRAIN_TOKENS],
    "split_leakage": "identical records in two splits are rejected",
    "duplicates": "exact duplicates inside a split are reported, not removed",
    "preprocessing": "chat template rendered by the base tokenizer; loss masked to assistant "
    f"tokens (label {IGNORE_INDEX} elsewhere); no truncation - over-length rows are rejected",
}


def _check_splits(
    splits: Mapping[str, Sequence[Mapping[str, Any]]],
    max_sequence_length: int,
    token_length: Callable[[Mapping[str, Any]], int] | None,
) -> dict[str, Any]:
    """Raise ValueError/TypeError naming the first violated rule; return the observations."""
    if not isinstance(splits, Mapping) or "train" not in splits:
        raise TypeError("splits must be a mapping with at least a 'train' split")
    if isinstance(max_sequence_length, bool) or not isinstance(max_sequence_length, int):
        raise TypeError("max_sequence_length must be an int")
    if not 1 <= max_sequence_length <= MAX_SEQUENCE_LENGTH_CEILING:
        raise ValueError(
            "max_sequence_length must be between 1 and "
            f"MAX_SEQUENCE_LENGTH_CEILING={MAX_SEQUENCE_LENGTH_CEILING}"
        )
    observed: dict[str, Any] = {"splits": {}, "duplicates": {}, "tokens": {}}
    prints: dict[str, set[str]] = {}
    for name, records in splits.items():
        if not isinstance(records, Sequence) or isinstance(records, str | bytes):
            raise TypeError(f"split {name!r} must be a sequence of records")
        if name == "train" and len(records) < MIN_TRAIN_EXAMPLES:
            raise ValueError(
                f"train split needs at least MIN_TRAIN_EXAMPLES={MIN_TRAIN_EXAMPLES} records"
            )
        fps = []
        for record in records:
            if not isinstance(record, Mapping) or "messages" not in record:
                raise TypeError(f"split {name!r}: every record must be canonical ({{'messages'}})")
            messages = record["messages"]
            if any(m.get("role") not in ALLOWED_ROLES for m in messages):
                raise ValueError(f"split {name!r}: invalid role in record")
            if not any(m["role"] == "assistant" and str(m["content"]).strip() for m in messages):
                raise ValueError(f"split {name!r}: record has no non-empty assistant turn")
            fps.append(fingerprint(record))
        prints[name] = set(fps)
        observed["splits"][name] = len(records)
        observed["duplicates"][name] = len(fps) - len(prints[name])
    for left, right in (("train", "validation"), ("train", "test"), ("validation", "test")):
        if left in prints and right in prints and prints[left] & prints[right]:
            raise ValueError(f"Split leakage: identical records in {left} and {right}")
    if token_length is not None:
        for name, records in splits.items():
            lengths = [int(token_length(r)) for r in records]
            longest = max(lengths, default=0)
            if longest > max_sequence_length:
                raise ValueError(
                    f"DATASET_SEQUENCE_TOO_LONG: split {name!r} has a {longest}-token example; "
                    f"max_sequence_length={max_sequence_length}"
                )
            observed["tokens"][name] = {"total": sum(lengths), "longest": longest}
        if observed["tokens"].get("train", {}).get("total", 0) > MAX_TOTAL_TRAIN_TOKENS:
            raise ValueError(
                f"DATASET_TOKEN_BUDGET_EXCEEDED: MAX_TOTAL_TRAIN_TOKENS={MAX_TOTAL_TRAIN_TOKENS}"
            )
    return observed


def validate_inputs(
    splits: Mapping[str, Sequence[Mapping[str, Any]]],
    max_sequence_length: int = DEFAULT_MAX_SEQUENCE_LENGTH,
    *,
    token_length: Callable[[Mapping[str, Any]], int] | None = None,
    names: Sequence[str] | None = None,
) -> dict[str, Any]:
    """Validation stage: return the input manifest (schema, per-split observations, verdict).

    Applies exactly the checks ``LanguageModelPipeline.prepare_splits`` applies before masking
    (schema, roles, assistant turns, leakage, duplicates, and - when ``token_length`` is given,
    normally ``pipe.token_length`` - the sequence and budget ceilings). Rejection is reported by
    raising exactly as the pipeline would; a caller that wants the finding recorded catches the
    exception and stores ``str(exc)`` under ``findings``.
    """
    observed = _check_splits(splits, max_sequence_length, token_length)
    if names is not None and len(names) != len(splits):
        raise ValueError("names must have one entry per split")
    return {
        "schema": dict(INPUT_SCHEMA),
        "inputs": [
            {
                "id": names[i] if names else name,
                "split": name,
                "records": observed["splits"][name],
                "duplicates": observed["duplicates"][name],
                "tokens": observed["tokens"].get(name),
            }
            for i, name in enumerate(splits)
        ],
        "max_sequence_length": max_sequence_length,
        "token_lengths_measured": token_length is not None,
        "dataset_digest": dataset_digest(splits),
        "verdict": "accepted",
        "findings": [
            {"input": name, "verdict": "accepted", "message": f"{n} exact duplicate(s) inside the split; none removed"}  # noqa: E501
            for name, n in observed["duplicates"].items()
            if n
        ],
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
    }


PROMPT_SCHEMA: dict[str, Any] = {
    "input": "one prompt string per request (rendered as a single user turn) or a list of "
    "{'role', 'content'} turns ending with a user turn",
    "prompt_chars": [1, MAX_PROMPT_CHARS],
    "max_new_tokens": [1, MAX_NEW_TOKENS_CEILING],
    "rendered_prompt_tokens": [1, MAX_SEQUENCE_LENGTH_CEILING],
    "decoding": "greedy (do_sample=False) unless sampling settings are passed explicitly",
}


def _check_prompt(prompt: Any, max_new_tokens: int) -> list[dict[str, str]]:
    """Raise TypeError/ValueError naming the first violated prompt rule; return the turns."""
    if isinstance(max_new_tokens, bool) or not isinstance(max_new_tokens, int):
        raise TypeError("max_new_tokens must be an int")
    if not 1 <= max_new_tokens <= MAX_NEW_TOKENS_CEILING:
        raise ValueError(
            f"max_new_tokens must be between 1 and MAX_NEW_TOKENS_CEILING={MAX_NEW_TOKENS_CEILING}"
        )
    if isinstance(prompt, str):
        messages = [{"role": "user", "content": prompt}]
    elif isinstance(prompt, Sequence) and not isinstance(prompt, bytes):
        messages = [dict(m) for m in prompt]
    else:
        raise TypeError("prompt must be a string or a sequence of {'role', 'content'} turns")
    if not messages or messages[-1].get("role") != "user":
        raise ValueError("a prompt must end with a user turn")
    for turn in messages:
        if turn.get("role") not in ALLOWED_ROLES:
            raise ValueError(f"invalid role {turn.get('role')!r}")
        content = turn.get("content")
        if not isinstance(content, str) or not content.strip():
            raise ValueError("every turn needs non-empty string content")
        if len(content) > MAX_PROMPT_CHARS:
            raise ValueError(f"prompt turn exceeds MAX_PROMPT_CHARS={MAX_PROMPT_CHARS} characters")
    return messages


def validate_prompts(
    prompts: Sequence[Any],
    max_new_tokens: int = 96,
    *,
    token_length: Callable[[Sequence[Mapping[str, str]]], int] | None = None,
    names: Sequence[str] | None = None,
) -> dict[str, Any]:
    """Validation stage for inference requests: the input manifest for a list of prompts.

    Applies exactly the checks ``LanguageModelPipeline.generate`` applies (turn shape, roles,
    non-empty content, character and ``max_new_tokens`` ceilings; the rendered-token ceiling when
    ``token_length`` - normally ``pipe.prompt_token_length`` - is given). Raises as ``generate``
    would; a caller records the message under ``findings``."""
    if not isinstance(prompts, Sequence) or isinstance(prompts, str | bytes):
        raise TypeError("prompts must be a sequence")
    if not prompts:
        raise ValueError("prompts must not be empty")
    checked = [_check_prompt(p, max_new_tokens) for p in prompts]
    if names is not None and len(names) != len(checked):
        raise ValueError("names must have one entry per prompt")
    inputs = []
    for i, messages in enumerate(checked):
        entry: dict[str, Any] = {
            "id": names[i] if names else f"prompt-{i}",
            "turns": len(messages),
            "chars": sum(len(m["content"]) for m in messages),
        }
        if token_length is not None:
            tokens = int(token_length(messages))
            if tokens > MAX_SEQUENCE_LENGTH_CEILING:
                raise ValueError(
                    f"rendered prompt has {tokens} tokens; the ceiling is "
                    f"MAX_SEQUENCE_LENGTH_CEILING={MAX_SEQUENCE_LENGTH_CEILING}"
                )
            entry["rendered_tokens"] = tokens
        inputs.append(entry)
    return {
        "schema": dict(PROMPT_SCHEMA),
        "inputs": inputs,
        "max_new_tokens": max_new_tokens,
        "verdict": "accepted",
        "findings": [],
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
    }


# ---------------------------------------------------------------------------
# Chat rendering, assistant-only masking, loss (§20.7)
# ---------------------------------------------------------------------------


def render_chat(tokenizer: Any, messages: Sequence[Mapping[str, str]], add_generation_prompt: bool = False) -> str:  # noqa: E501
    """Render turns with the tokenizer's own chat template. ``enable_thinking=False`` asks
    Qwen3-style templates for a direct answer; templates without that switch ignore it. Templates
    with no ``system`` role (danube3) get the system text folded into the first user turn."""
    messages = [dict(m) for m in messages]
    template = getattr(tokenizer, "chat_template", None) or ""
    if messages and messages[0]["role"] == "system" and "system" not in template:
        system, rest = messages[0], messages[1:]
        if rest and rest[0]["role"] == "user":
            rest[0] = {"role": "user", "content": f"{system['content']}\n\n{rest[0]['content']}"}
            messages = rest
    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=add_generation_prompt, enable_thinking=False
    )


def assistant_char_spans(tokenizer: Any, messages: Sequence[Mapping[str, str]], full_text: str) -> list[tuple[int, int]]:  # noqa: E501
    """Character ranges ``[start, end)`` of each assistant turn inside the rendered conversation."""
    end_of_turn = getattr(tokenizer, "eos_token", None) or "<|im_end|>"
    last_assistant = max(i for i, m in enumerate(messages) if m["role"] == "assistant")
    spans = []
    cursor = 0
    for index, message in enumerate(messages):
        content = message["content"].strip()
        if message["role"] != "assistant":
            found = full_text.find(content, cursor)
            if found != -1:
                cursor = found + len(content)
            continue
        start = -1
        if index == last_assistant:
            generation_prompt = render_chat(tokenizer, messages[:index], add_generation_prompt=True)
            if full_text.startswith(generation_prompt):
                start = len(generation_prompt)
        if start == -1:
            start = full_text.find(content, cursor)
            if start == -1:
                raise ValueError(f"Assistant content not found in rendered text: {content[:60]!r}")
        eos_at = full_text.find(end_of_turn, start)
        end = eos_at + len(end_of_turn) if eos_at != -1 else len(full_text)
        spans.append((start, end))
        cursor = end
    return spans


def build_masked_example(tokenizer: Any, record: Mapping[str, Any], max_sequence_length: int = DEFAULT_MAX_SEQUENCE_LENGTH) -> tuple[list[int], list[int]]:  # noqa: E501
    """Return ``(input_ids, labels)``, labels ``IGNORE_INDEX`` everywhere except assistant turns.

    Over-length examples are rejected, never truncated (a truncated answer would teach the model
    to stop mid-sentence)."""
    messages = record["messages"]
    full_text = render_chat(tokenizer, messages)

    def encode(text: str) -> list[int]:
        return list(tokenizer(text, add_special_tokens=False)["input_ids"])

    try:
        encoded = tokenizer(full_text, add_special_tokens=False, return_offsets_mapping=True)
        input_ids, offsets = list(encoded["input_ids"]), list(encoded["offset_mapping"])
    except Exception:  # slow tokenizers cannot report offsets; fall back to prefix rendering
        input_ids, offsets = encode(full_text), None
    if len(input_ids) > max_sequence_length:
        raise ValueError("DATASET_SEQUENCE_TOO_LONG")

    labels = [IGNORE_INDEX] * len(input_ids)
    if offsets is not None:
        for start, end in assistant_char_spans(tokenizer, messages, full_text):
            for position, (char_start, char_end) in enumerate(offsets):
                inside_turn = start <= char_start and char_end <= end
                if inside_turn and char_start < char_end:
                    labels[position] = input_ids[position]
    else:
        for index, message in enumerate(messages):
            if message["role"] != "assistant":
                continue
            before = encode(render_chat(tokenizer, messages[:index], add_generation_prompt=True))
            through = encode(render_chat(tokenizer, messages[: index + 1]))
            if input_ids[: len(before)] != before or input_ids[: len(through)] != through:
                raise ValueError("Chat template rendering is not prefix-stable for this tokenizer")
            labels[len(before) : len(through)] = input_ids[len(before) : len(through)]

    if all(label == IGNORE_INDEX for label in labels):
        raise ValueError("No supervised tokens in example")
    return input_ids, labels


def show_supervision(tokenizer: Any, input_ids: Sequence[int], labels: Sequence[int]) -> str:
    """Decode an example, marking supervised token runs with ⟦ ⟧."""
    pieces = []
    inside = False
    for token_id, label in zip(input_ids, labels, strict=True):
        supervised = label != IGNORE_INDEX
        if supervised != inside:
            pieces.append("⟦" if supervised else "⟧")
            inside = supervised
        pieces.append(tokenizer.decode([token_id]))
    if inside:
        pieces.append("⟧")
    return "".join(pieces)


def supervised_token_count(labels: Sequence[int]) -> int:
    return sum(1 for label in labels if label != IGNORE_INDEX)


def perplexity(loss: float | None) -> float | None:
    """exp(loss); None when the loss is missing or too large to be meaningful (>= 20 nats)."""
    if loss is None or loss >= 20:
        return None
    return math.exp(loss)


# ---------------------------------------------------------------------------
# Adapter-bundle contract (§16, §18): export, safe extraction, verification
# ---------------------------------------------------------------------------


def safe_member_path(root: str | Path, member_name: str) -> Path:
    """Resolve an archive member name under ``root``, refusing anything that could escape it."""
    if "\\" in member_name:
        raise ValueError(f"Unsafe archive path (backslash): {member_name!r}")
    relative = PurePosixPath(member_name)
    if relative.is_absolute() or ".." in relative.parts:
        raise ValueError(f"Unsafe archive path: {member_name!r}")
    root = Path(root).resolve()
    target = (root / Path(*relative.parts)).resolve()
    if target != root and root not in target.parents:
        raise ValueError(f"Archive member escapes the extraction root: {member_name!r}")
    return target


def extract_zip_safely(zip_path: str | Path, root: str | Path, size_limit_bytes: int) -> Path:
    """Extract into ``root``, refusing symlinks, path escapes and archives that expand past the
    limit. Members are copied one by one; ``extractall`` is never used."""
    root = Path(root).resolve()
    shutil.rmtree(root, ignore_errors=True)
    root.mkdir(parents=True)
    expanded = 0
    with zipfile.ZipFile(zip_path) as archive:
        for info in archive.infolist():
            if stat.S_ISLNK((info.external_attr >> 16) & 0xFFFF):
                raise ValueError("Symlinks are not allowed in the archive")
            expanded += info.file_size
            if expanded > size_limit_bytes:
                raise ValueError("Archive expands beyond the allowed size")
            target = safe_member_path(root, info.filename)
            if info.is_dir():
                target.mkdir(parents=True, exist_ok=True)
                continue
            target.parent.mkdir(parents=True, exist_ok=True)
            with archive.open(info) as source, open(target, "wb") as destination:
                shutil.copyfileobj(source, destination)
    return root


def write_artifact_manifest(bundle_dir: str | Path) -> dict[str, Any]:
    """List every bundle file with its size and SHA-256 into artifact-manifest.json."""
    bundle = Path(bundle_dir)
    records = [
        {
            "path": path.relative_to(bundle).as_posix(),
            "bytes": path.stat().st_size,
            "sha256": sha256_of_file(path),
        }
        for path in sorted(bundle.rglob("*"))
        if path.is_file() and path.name != ARTIFACT_MANIFEST_NAME
    ]
    manifest = {
        "format": ARTIFACT_FORMAT,
        "formatVersion": ARTIFACT_FORMAT_VERSION,
        "files": records,
        "totalBytes": sum(r["bytes"] for r in records),
    }
    (bundle / ARTIFACT_MANIFEST_NAME).write_text(json.dumps(manifest, indent=2), encoding="utf-8")
    return manifest


def verify_artifact_bundle(bundle_dir: str | Path) -> tuple[dict[str, Any], dict[str, Any]]:
    """Validate an adapter bundle BEFORE any model state is deserialised (AINF3/AINF4).

    Checks the manifest format, every listed file's size and SHA-256, that no unlisted file is
    present, and that ``provenance.json`` names this module's pinned base model at its 40-hex
    revision with ``trustRemoteCode`` false. Returns ``(manifest, provenance)``."""
    bundle = Path(bundle_dir).resolve()
    manifest_path = bundle / ARTIFACT_MANIFEST_NAME
    if not manifest_path.is_file():
        raise FileNotFoundError(f"artifact manifest not found: {manifest_path}")
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    if manifest.get("format") != ARTIFACT_FORMAT or manifest.get("formatVersion") != ARTIFACT_FORMAT_VERSION:  # noqa: E501
        raise ValueError("Unsupported artifact format")
    listed, listed_bytes = set(), 0
    for record in manifest.get("files", []):
        file_path = safe_member_path(bundle, record["path"])
        if not file_path.is_file() or file_path.stat().st_size != record["bytes"] or sha256_of_file(file_path) != record["sha256"]:  # noqa: E501
            raise ValueError(f"SHA-256 mismatch: {record['path']}")
        listed.add(record["path"])
        listed_bytes += record["bytes"]
    on_disk = {
        p.relative_to(bundle).as_posix()
        for p in bundle.rglob("*")
        if p.is_file() and p.name != ARTIFACT_MANIFEST_NAME
    }
    if on_disk != listed or listed_bytes != manifest.get("totalBytes"):
        raise ValueError("Manifest/file-set mismatch: files were added, removed, or the total size differs")  # noqa: E501
    for required in ("adapter_config.json", "adapter_model.safetensors", "provenance.json", "tokenizer/tokenizer_config.json"):  # noqa: E501
        if required not in listed:
            raise ValueError(f"artifact bundle is missing {required}")
    provenance = json.loads((bundle / "provenance.json").read_text(encoding="utf-8"))
    if provenance.get("trustRemoteCode") is not False:
        raise ValueError("trustRemoteCode must be false")
    if provenance.get("artifactFormat") != ARTIFACT_FORMAT:
        raise ValueError("provenance artifactFormat must be peft_adapter")
    revision = provenance.get("baseModelRevision") or ""
    if len(revision) != 40 or any(c not in "0123456789abcdef" for c in revision):
        raise ValueError("Invalid baseModelRevision provenance: expected a 40-character commit SHA, not a branch or tag")  # noqa: E501
    if (provenance.get("baseModel"), revision) != (MODEL_ID, MODEL_REVISION):
        raise ValueError(
            f"artifact was trained on {provenance.get('baseModel')}@{revision[:12]}, "
            f"this pipeline pins {MODEL_ID}@{MODEL_REVISION[:12]}; refusing to attach"
        )
    return manifest, provenance


# ---------------------------------------------------------------------------
# Evaluation report (EVAL21)
# ---------------------------------------------------------------------------


def evaluation_report(
    metrics: Mapping[str, Any] | None,
    *,
    sample_kind: str = "sample",
    n_train: int = 0,
    n_validation: int = 0,
    probes: Sequence[Mapping[str, str]] | None = None,
) -> dict[str, Any]:
    """Evaluation stage: a machine-readable report even when nothing is measurable.

    ``metrics`` is the dict the tutorial's training loop produces (``trainLoss``,
    ``validationLoss``, ``validationPerplexity`` - the ids the repository's model card and result
    contract use). They are optimisation evidence on a manufactured validation split, so the best
    verdict is ``sample-sanity``; task quality stays ``not-measurable`` without a held-out task
    set. ``probes`` (base vs adapted answers) are recorded as qualitative evidence only."""
    base = {
        "task": "language-model supervised fine-tuning (QLoRA adapter)",
        "score_semantics": "mean cross-entropy per supervised assistant token; perplexity = exp(loss)",  # noqa: E501
        "decoding_rule": DECODING_RULE,
        "sample_kind": sample_kind,
        "n_train": int(n_train),
        "n_validation": int(n_validation),
        "baselines": [],
        "probes": [dict(p) for p in probes] if probes else [],
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
    }
    needs = (
        "a held-out evaluation set from the target task with a rubric or human ratings (and a "
        "pre-adaptation baseline scored the same way) for any task-quality claim; validation loss "
        "and perplexity only show that the adapter fits the format of the training distribution"
    )
    validation_loss = None if metrics is None else metrics.get("validationLoss")
    if validation_loss is None:
        return {
            **base,
            "metrics": [],
            "verdict": "not-measurable",
            "reason": "no labelled held-out split was scored",
            "needs": needs,
        }
    entries = []
    for key in ("trainLoss", "validationLoss", "testLoss", "validationPerplexity"):
        value = metrics.get(key)
        if value is not None:
            entries.append(
                {
                    "id": key,
                    "value": float(value),
                    "units": "nats per supervised token" if key.endswith("Loss") else "tokens",
                    "estimation": "single seeded run on a manufactured validation split; no dispersion estimate",  # noqa: E501
                }
            )
    return {
        **base,
        "metrics": entries,
        "verdict": "sample-sanity",
        "reason": (
            f"optimisation metrics on {n_validation} held-out row(s) manufactured from the "
            "training source; they measure format fit, not task quality"
        ),
        "needs": needs,
    }


# ---------------------------------------------------------------------------
# Pipeline
# ---------------------------------------------------------------------------


@dataclass
class LanguageModelPipeline:
    """Tokenizer + base causal LM loaded from a digest-verified snapshot.

    ``model`` is the base ``AutoModelForCausalLM`` (4-bit ``nf4`` when a CUDA device is present
    and ``quantize_4bit`` is not False); the tutorial wraps it with PEFT for training and reload.
    ``generate`` and ``evaluate_loss`` accept a different model object (the PEFT-wrapped one) so the
    same helpers serve base, adapted and reloaded models."""

    tokenizer: Any
    model: Any
    device: str = "cpu"
    compute_dtype: str = "float32"
    quantized: bool = False
    source: str = "local-snapshot"
    weights_dir: str = ""
    load_kwargs: dict[str, Any] = field(default_factory=dict)

    @classmethod
    def from_pretrained(
        cls,
        device: str | None = None,
        weights_dir: str | Path | None = None,
        allow_download: bool = False,
        *,
        quantize_4bit: bool | None = None,
    ) -> LanguageModelPipeline:
        import torch
        from transformers import AutoModelForCausalLM, AutoTokenizer

        root = Path(weights_dir or DEFAULT_WEIGHTS_DIR)
        stage_missing_files(root, allow_download=allow_download)
        verify_snapshot(root)
        tokenizer = AutoTokenizer.from_pretrained(str(root), local_files_only=True, trust_remote_code=False)  # noqa: E501
        if tokenizer.pad_token_id is None:
            tokenizer.pad_token = tokenizer.eos_token
        if not tokenizer.chat_template:
            raise ValueError("This tokenizer ships no chat template; the pipeline needs one to render turns")  # noqa: E501
        resolved_device = device or ("cuda:0" if torch.cuda.is_available() else "cpu")
        on_cuda = resolved_device.startswith("cuda")
        quantized = on_cuda if quantize_4bit is None else (quantize_4bit and on_cuda)
        compute_dtype = (torch.bfloat16 if on_cuda and torch.cuda.is_bf16_supported() else torch.float16) if on_cuda else torch.float32  # noqa: E501
        kwargs: dict[str, Any] = {"trust_remote_code": False, "local_files_only": True, "dtype": compute_dtype, "attn_implementation": "sdpa"}  # noqa: E501
        if on_cuda:
            kwargs["device_map"] = {"": 0}
        if quantized:
            from transformers import BitsAndBytesConfig

            kwargs["quantization_config"] = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_quant_type="nf4",
                bnb_4bit_use_double_quant=True,
                bnb_4bit_compute_dtype=compute_dtype,
            )
        model = AutoModelForCausalLM.from_pretrained(str(root), **kwargs)
        if not on_cuda:
            model = model.to(resolved_device)
        model.eval()
        return cls(tokenizer, model, resolved_device, str(compute_dtype).replace("torch.", ""), quantized, "local-snapshot", str(root), kwargs)  # noqa: E501

    def reload_base(self) -> Any:
        """Load the base model again from the same verified snapshot (fresh reload proof)."""
        from transformers import AutoModelForCausalLM

        model = AutoModelForCausalLM.from_pretrained(self.weights_dir, **self.load_kwargs)
        if not self.device.startswith("cuda"):
            model = model.to(self.device)
        return model.eval()

    # -- data -------------------------------------------------------------------------------

    def render_chat(self, messages: Sequence[Mapping[str, str]], add_generation_prompt: bool = False) -> str:  # noqa: E501
        return render_chat(self.tokenizer, messages, add_generation_prompt)

    def token_length(self, record: Mapping[str, Any]) -> int:
        text = self.render_chat(record["messages"])
        return len(self.tokenizer(text, add_special_tokens=False)["input_ids"])

    def prompt_token_length(self, messages: Sequence[Mapping[str, str]]) -> int:
        text = self.render_chat(messages, add_generation_prompt=True)
        return len(self.tokenizer(text, add_special_tokens=False)["input_ids"])

    def prepare_splits(
        self,
        splits: Mapping[str, Sequence[Mapping[str, Any]]],
        max_sequence_length: int = DEFAULT_MAX_SEQUENCE_LENGTH,
    ) -> dict[str, list[tuple[list[int], list[int]]]]:
        """Validate the splits exactly as ``validate_inputs`` does, then mask every example."""
        _check_splits(splits, max_sequence_length, self.token_length)
        return {
            name: [build_masked_example(self.tokenizer, r, max_sequence_length) for r in records]
            for name, records in splits.items()
        }

    # -- inference ----------------------------------------------------------------------------

    def _tensor_device(self, model: Any) -> Any:
        try:
            return next(model.parameters()).device
        except StopIteration:  # pragma: no cover - a model without parameters
            return self.device

    def generate(self, prompt: str | Sequence[Mapping[str, str]], *, model: Any = None, max_new_tokens: int = 96, **decoding: Any) -> str:  # noqa: E501
        """Eval-mode generation (greedy unless ``decoding`` says otherwise); restores the model's
        training state afterwards so it is safe to call mid-training."""
        import torch

        model = self.model if model is None else model
        messages = _check_prompt(prompt, max_new_tokens)
        was_training = getattr(model, "training", False)
        model.eval()
        previous_use_cache = getattr(model.config, "use_cache", True)
        model.config.use_cache = True
        prompt_text = self.render_chat(messages, add_generation_prompt=True)
        inputs = self.tokenizer(prompt_text, return_tensors="pt", add_special_tokens=False).to(self._tensor_device(model))  # noqa: E501
        settings = {"do_sample": False, **decoding}
        with torch.inference_mode():
            output_ids = model.generate(**inputs, max_new_tokens=max_new_tokens, pad_token_id=self.tokenizer.pad_token_id, **settings)  # noqa: E501
        model.config.use_cache = previous_use_cache
        if was_training:
            model.train()
        return self.tokenizer.decode(output_ids[0, inputs["input_ids"].shape[1] :], skip_special_tokens=True).strip()  # noqa: E501

    def to_batch(self, example: tuple[Sequence[int], Sequence[int]], model: Any = None) -> dict[str, Any]:  # noqa: E501
        import torch

        device = self._tensor_device(self.model if model is None else model)
        input_ids, labels = example
        return {
            "input_ids": torch.tensor([list(input_ids)], device=device),
            "labels": torch.tensor([list(labels)], device=device),
            "attention_mask": torch.ones((1, len(input_ids)), dtype=torch.long, device=device),
        }

    def evaluate_loss(self, examples: Sequence[tuple[Sequence[int], Sequence[int]]], *, model: Any = None) -> float | None:  # noqa: E501
        """Mean cross-entropy per supervised token over a split, in eval mode with gradients off."""
        import torch

        model = self.model if model is None else model
        if not examples:
            return None
        was_training = getattr(model, "training", False)
        model.eval()
        total_loss, total_tokens = 0.0, 0
        with torch.inference_mode():
            for example in examples:
                batch = self.to_batch(example, model)
                count = supervised_token_count(list(example[1]))
                total_loss += float(model(**batch).loss.item()) * count
                total_tokens += count
        if was_training:
            model.train()
        return total_loss / total_tokens if total_tokens else None

    # -- artifact -------------------------------------------------------------------------

    def export_adapter_bundle(self, peft_model: Any, bundle_dir: str | Path, *, metrics: Mapping[str, Any], provenance: Mapping[str, Any]) -> dict[str, Any]:  # noqa: E501
        """Write the adapter, the tokenizer (with its chat template), metrics, provenance and the
        artifact manifest. The bundle contains no training rows and no base weights."""
        bundle = Path(bundle_dir)
        shutil.rmtree(bundle, ignore_errors=True)
        bundle.mkdir(parents=True)
        peft_model.save_pretrained(str(bundle), safe_serialization=True)
        self.tokenizer.save_pretrained(str(bundle / "tokenizer"))
        full_provenance = {
            "artifactFormat": ARTIFACT_FORMAT,
            "artifactFormatVersion": ARTIFACT_FORMAT_VERSION,
            "baseModel": MODEL_ID,
            "baseModelRevision": MODEL_REVISION,
            "baseModelLicense": MODEL_LICENSE,
            "modelKey": MODEL_KEY,
            "trustRemoteCode": False,
            "quantized": self.quantized,
            **dict(provenance),
        }
        (bundle / "metrics.json").write_text(json.dumps(dict(metrics), indent=2), encoding="utf-8")
        (bundle / "provenance.json").write_text(json.dumps(full_provenance, indent=2), encoding="utf-8")  # noqa: E501
        (bundle / "MODEL_CARD.md").write_text(
            f"# PEFT adapter for {MODEL_ID}\n\nBase revision: `{MODEL_REVISION}`. "
            f"Dataset digest: `{full_provenance.get('datasetDigest', 'unknown')}`.\n"
            "Optimization metrics in metrics.json are not task-quality evidence.\n",
            encoding="utf-8",
        )
        return write_artifact_manifest(bundle)

    def load_adapter(self, bundle_dir: str | Path, *, base_model: Any = None) -> Any:
        """Verify a bundle, then attach its adapter to a base model in eval mode (AINF5: the base
        weights come from the digest-verified snapshot, never from the bundle or the network)."""
        from peft import PeftModel
        from transformers import AutoTokenizer

        bundle = Path(bundle_dir).resolve()
        verify_artifact_bundle(bundle)
        base = self.model if base_model is None else base_model
        self.tokenizer = AutoTokenizer.from_pretrained(str(bundle / "tokenizer"), local_files_only=True, trust_remote_code=False)  # noqa: E501
        if self.tokenizer.pad_token_id is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token
        return PeftModel.from_pretrained(base, str(bundle), is_trainable=False).eval()

## 3. Pin, stage and verify the model

The model identity is carried twice — `MODEL_ID`/`MODEL_REVISION` in the module above and the `13`-file manifest below (paths, byte sizes, SHA-256) — and the cell first asserts they agree. It writes the manifest into the working-directory snapshot, then `stage_missing_files(..., allow_download=True)` fetches exactly the entries that are absent from the Hugging Face Hub **at revision `a10cc1512eab…`** (never `main`), `verify_snapshot` re-hashes every file and raises on the first size or digest mismatch, and only then does `LanguageModelPipeline.from_pretrained(weights_dir=WEIGHTS_DIR)` load the verified files. There is no fallback to a different download and no remote model code is executed. The effective identity, device and weight source are printed before any inference.

In [ ]:
import json

MANIFEST = {
  "format": "dimer_hf_snapshot",
  "formatVersion": 1,
  "modelKey": "smollm2-360m",
  "modelId": "HuggingFaceTB/SmolLM2-360M-Instruct",
  "revision": "a10cc1512eabd3dde888204e902eca88bddb4951",
  "files": [
    {
      "path": "all_results.json",
      "bytes": 786,
      "sha256": "71face5e244d784819f937a25206865c295c8889fe4359eec28bbec85d02da0d"
    },
    {
      "path": "config.json",
      "bytes": 846,
      "sha256": "224f72354f10d617a359cc82ad15a3c96e866b9b2ffadb81997eeea9e88e22ee"
    },
    {
      "path": "eval_results.json",
      "bytes": 589,
      "sha256": "db9bcb5c4f2a3be0d7e88f86af26f375c22c240954af86942d1554d87a488ce8"
    },
    {
      "path": "generation_config.json",
      "bytes": 132,
      "sha256": "87b916edaaab66b3899b9d0dd0752727dff6666686da0504d89ae0a6e055a013"
    },
    {
      "path": "merges.txt",
      "bytes": 466391,
      "sha256": "0b54e8aa4e53d5383e2e4bc635a56b43f9647f7b13832d5d9ecd8f82dac4f510"
    },
    {
      "path": "model.safetensors",
      "bytes": 723674912,
      "sha256": "e6bffe7435d7ddc10fd3b9a9efd429dafbacb1cb17015fb5562664e7532bf86e"
    },
    {
      "path": "README.md",
      "bytes": 7304,
      "sha256": "6b88794416ac9da8f254ebb0bec228967a2bdd0badf9a2853863928b25facd95"
    },
    {
      "path": "special_tokens_map.json",
      "bytes": 655,
      "sha256": "2b7379f3ae813529281a5c602bc5a11c1d4e0a99107aaa597fe936c1e813ca52"
    },
    {
      "path": "tokenizer.json",
      "bytes": 2104556,
      "sha256": "9ca9acddb6525a194ec8ac7a87f24fbba7232a9a15ffa1af0c1224fcd888e47c"
    },
    {
      "path": "tokenizer_config.json",
      "bytes": 3764,
      "sha256": "4ec77d44f62efeb38d7e044a1db318f6a939438425312dfa333b8382dbad98df"
    },
    {
      "path": "train_results.json",
      "bytes": 232,
      "sha256": "fceae2634b0a268a8adbda73fd0b2192922b496054011f588d2c6e888db1cea1"
    },
    {
      "path": "trainer_state.json",
      "bytes": 57621,
      "sha256": "2263f580ae0d7890acc11253b83138b3fc1250dc6728688a3387ce813b92eed1"
    },
    {
      "path": "vocab.json",
      "bytes": 800662,
      "sha256": "82b84012e3add4d01d12ba14442026e49b8cbbaead1f79ecf3d919784f82dc79"
    }
  ],
  "totalBytes": 727118450
}

if (MANIFEST['modelId'], MANIFEST['revision']) != (MODEL_ID, MODEL_REVISION):
    raise RuntimeError('inline manifest does not name the identity carried by the pipeline module; the notebook was not regenerated after a change')
WEIGHTS_DIR = DEFAULT_WEIGHTS_DIR
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
with open(WEIGHTS_DIR / MANIFEST_NAME, 'w', encoding='utf-8') as handle:
    json.dump(MANIFEST, handle, indent=2)
print({'model_id': MODEL_ID, 'revision': MODEL_REVISION, 'license': MODEL_LICENSE, 'files': len(MANIFEST['files']), 'total_bytes': MANIFEST['totalBytes']})
fetched = stage_missing_files(WEIGHTS_DIR, allow_download=True)
print({'weights_dir': str(WEIGHTS_DIR), 'fetched': fetched})
snapshot = verify_snapshot(WEIGHTS_DIR)
_files = snapshot.get('files', []) if isinstance(snapshot, dict) else []
print({'verified_files': len(_files) if isinstance(_files, list) else _files, 'revision': snapshot.get('revision', MODEL_REVISION) if isinstance(snapshot, dict) else MODEL_REVISION})
pipe = LanguageModelPipeline.from_pretrained(weights_dir=WEIGHTS_DIR)
print({'device': getattr(pipe, 'device', None), 'source': getattr(pipe, 'source', 'local-snapshot')})

## 4. Supply the external adapter bundle and verify it before any model state is loaded

Leave `ARTIFACT_DIR` empty to upload the bundle ZIP; set it to a directory already in the runtime to skip the dialog (an executor places the files there). An uploaded archive is extracted with `extract_zip_safely`: every member must be a relative path without `..`, a leading `/` or backslashes that resolves inside the extraction folder; symlinks are refused; the archive may not expand past 512 MiB; `extractall` is never used. If the sender gave you the whole-archive SHA-256, paste it into `EXPECTED_ARTIFACT_ZIP_SHA256` and a substituted archive fails before extraction.

`verify_artifact_bundle` then runs **before** any state is deserialised: exactly the manifest format the module writes, every listed file present with its recorded size and SHA-256, no file on disk the manifest does not list, the required members (`adapter_config.json`, `adapter_model.safetensors`, `provenance.json`, the tokenizer), `trustRemoteCode` false, a 40-character `baseModelRevision` (a branch name like `main` is rejected — the Hub can move it), and the base model and revision equal to the ones carried by the module. A mismatch stops the notebook: an adapter attached to weights it was not trained on produces confident nonsense rather than an error. The cell prints the provenance a consumer needs: format, base model and revision, dataset digest and licence, training hyperparameters and the producer's runtime.

In [ ]:
ARTIFACT_DIR = ''  # @param {type:"string"}
EXPECTED_ARTIFACT_ZIP_SHA256 = ''  # @param {type:"string"}
os.makedirs('outputs', exist_ok=True)
if ARTIFACT_DIR:
    bundle_dir = Path(ARTIFACT_DIR)
    artifact_source = f'directory: {bundle_dir}'
    archive_sha = None
else:
    from google.colab import files
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError('Upload exactly one adapter bundle ZIP')
    archive_path = Path('work') / Path(next(iter(uploaded))).name
    archive_path.parent.mkdir(parents=True, exist_ok=True)
    archive_path.write_bytes(next(iter(uploaded.values())))
    archive_sha = sha256_of_file(archive_path)
    if EXPECTED_ARTIFACT_ZIP_SHA256 and archive_sha.lower() != EXPECTED_ARTIFACT_ZIP_SHA256.strip().lower():
        raise ValueError('Whole-ZIP SHA-256 mismatch: this is not the archive you were told to expect')
    print(f'archive {archive_path.name}: {archive_path.stat().st_size / 1024**2:.1f} MB, sha256 {archive_sha}')
    extraction_root = extract_zip_safely(archive_path, Path('work') / 'external-artifact', size_limit_bytes=512 * 1024**2)
    manifests = list(extraction_root.rglob(ARTIFACT_MANIFEST_NAME))
    if len(manifests) != 1:
        raise ValueError(f'Expected exactly one {ARTIFACT_MANIFEST_NAME} in the archive')
    bundle_dir = manifests[0].parent
    artifact_source = 'upload dialog'
artifact_manifest, provenance = verify_artifact_bundle(bundle_dir)
print({'artifact_source': artifact_source, 'format': artifact_manifest['format'], 'formatVersion': artifact_manifest['formatVersion'], 'files': len(artifact_manifest['files']), 'total_bytes': artifact_manifest['totalBytes']})
print({'baseModel': provenance['baseModel'], 'baseModelRevision': provenance['baseModelRevision'], 'baseModelLicense': provenance.get('baseModelLicense'), 'quantized': provenance.get('quantized'), 'datasetDigest': provenance.get('datasetDigest'), 'dataset': provenance.get('dataset')})
print({'training': provenance.get('training'), 'producer_runtime': provenance.get('runtime')})

## 5. Attach the adapter to the verified base model

`pipe.load_adapter` re-verifies the bundle, loads the tokenizer **from the bundle** (so prompts render with exactly the chat template the adapter saw in training), and attaches the deltas to the base model that Section 3 loaded from the digest-verified snapshot with `PeftModel.from_pretrained(..., is_trainable=False)` in eval mode. There is **no network fallback**: the only acceptable base weights are the files named by the inline manifest. The cell prints the adapter configuration and confirms that the adapter's `B` matrices are not all zero — an untrained or mis-saved adapter would be indistinguishable from the base model.

In [ ]:
model = pipe.load_adapter(bundle_dir)
lora_b_matrices = [param for name, param in model.named_parameters() if 'lora_b' in name.lower()]
if not lora_b_matrices or max(p.abs().max().item() for p in lora_b_matrices) == 0:
    raise RuntimeError('Adapter weights are zero or missing')
adapter_config = json.loads((Path(bundle_dir) / 'adapter_config.json').read_text(encoding='utf-8'))
print({'adapter_attached': True, 'r': adapter_config.get('r'), 'lora_alpha': adapter_config.get('lora_alpha'), 'target_modules': adapter_config.get('target_modules'), 'device': pipe.device, 'quantized_4bit': pipe.quantized, 'source': pipe.source})

## 6. Validate new prompts → input manifest

`validate_prompts` is the pipeline's public validation stage for inference requests: it applies exactly the checks `generate` applies — each prompt a non-empty user turn (or a list of turns ending with one), turn content within `MAX_PROMPT_CHARS`, `max_new_tokens` within `MAX_NEW_TOKENS_CEILING`, and the rendered prompt within `MAX_SEQUENCE_LENGTH_CEILING` tokens measured with the bundle's tokenizer — and returns an **input manifest** naming the schema, ceilings, per-prompt turn/character/token counts and the verdict. It is written to `outputs/language_model_artifact_inference_input_manifest.json`. To show what rejection looks like, the cell also validates a blank prompt and records the pipeline's own error message as a finding. Type your own prompt into `CUSTOM_PROMPT`; the two prefilled prompts are new to the adapter (they were not in the tutorial's training sample).

In [ ]:
CUSTOM_PROMPT = ''  # @param {type:"string"}
MAX_NEW_TOKENS = 96  # @param {type:"integer"}
PROMPTS = [
    'Ipaliwanag sa simpleng Filipino kung ano ang machine learning.',
    'Sumulat ng maikling payo para sa isang estudyanteng nagsisimula sa AI.',
]
if CUSTOM_PROMPT.strip():
    PROMPTS.append(CUSTOM_PROMPT.strip())
print({'ceilings': {'MAX_NEW_TOKENS_CEILING': MAX_NEW_TOKENS_CEILING, 'MAX_PROMPT_CHARS': MAX_PROMPT_CHARS, 'MAX_SEQUENCE_LENGTH_CEILING': MAX_SEQUENCE_LENGTH_CEILING}, 'decoding_rule': DECODING_RULE})
input_manifest = validate_prompts(PROMPTS, MAX_NEW_TOKENS, token_length=pipe.prompt_token_length, names=[f'prompt-{i}' for i in range(len(PROMPTS))])
# Demonstrate rejection on a blank prompt; the finding is recorded, not swallowed.
try:
    validate_prompts(['   '], MAX_NEW_TOKENS)
except ValueError as exc:
    input_manifest['findings'].append({'input': 'blank-prompt-probe', 'verdict': 'rejected', 'message': str(exc)})
with open('outputs/language_model_artifact_inference_input_manifest.json', 'w', encoding='utf-8') as handle:
    json.dump(input_manifest, handle, indent=2, ensure_ascii=False)
print(json.dumps(input_manifest, indent=2))

## 7. Generate with the adapter off and on, then with sampling; report what cannot be measured; export

The first comparison uses **greedy decoding** (`do_sample=False`, the module's `DECODING_RULE`) and non-thinking mode so that the result is deterministic and the only difference between the two columns is the adapter itself: `model.disable_adapter()` switches it off for the base answer. Expect the answers to be similar in substance and to differ in language, tone or format — that is what a small adapter trained on ~100 rows changes. Greedy decoding is right for a reproducible check and wrong for a product: small models fall into repetition loops under it, so the cell re-asks the first prompt twice with the direct-answer sampling settings (`do_sample=True, temperature=0.7, top_p=0.8, top_k=20`); each sampled answer differs and none should loop.

`evaluation_report` is the pipeline's public evaluation stage and is produced even here: with no labelled prompts its verdict is `not-measurable` and it states what would make the task measurable; it is written to `outputs/language_model_artifact_inference_evaluation_report.json`. The result JSON records the artifact identity and digests, the provenance the bundle carried, every prompt with its base, adapted and sampled answers, the input manifest, the notebook's source, the model identity, licence and runtime; the CSV keeps one row per prompt and answer kind. No credentials are recorded.

In [ ]:
import csv

def base_and_adapted(prompt):
    with model.disable_adapter():
        base_answer = pipe.generate(prompt, model=model, max_new_tokens=MAX_NEW_TOKENS)
    return base_answer, pipe.generate(prompt, model=model, max_new_tokens=MAX_NEW_TOKENS)

rows = []
for prompt in PROMPTS:
    base_answer, adapted_answer = base_and_adapted(prompt)
    rows.append({'prompt': prompt, 'base': base_answer, 'adapted': adapted_answer})
    print(f'PROMPT:  {prompt}\nBASE:    {base_answer}\nADAPTED: {adapted_answer}\n')
SAMPLING = {'do_sample': True, 'temperature': 0.7, 'top_p': 0.8, 'top_k': 20}
sampled = [pipe.generate(PROMPTS[0], model=model, max_new_tokens=MAX_NEW_TOKENS, **SAMPLING) for _ in range(2)]
for attempt, answer in enumerate(sampled, start=1):
    print(f'sampled answer {attempt}: {answer}\n')
report = evaluation_report(None, sample_kind='BYOD', probes=rows)
with open('outputs/language_model_artifact_inference_evaluation_report.json', 'w', encoding='utf-8') as handle:
    json.dump(report, handle, indent=2, ensure_ascii=False)
payload = {
    'artifact': {'source': artifact_source, 'bundle_dir': str(bundle_dir), 'zip_sha256': archive_sha, 'manifest': artifact_manifest, 'provenance': provenance, 'adapter_config': adapter_config},
    'evaluation_report': report,
    'input_manifest': input_manifest,
    'generations': rows,
    'sampled': {'prompt': PROMPTS[0], 'settings': SAMPLING, 'answers': sampled},
    'inference': {'decoding_rule': DECODING_RULE, 'max_new_tokens': MAX_NEW_TOKENS},
    'notebook_source': NOTEBOOK_SOURCE,
    'repository_revision': NOTEBOOK_SOURCE['repository_revision'],
    'model_id': MODEL_ID,
    'model_revision': MODEL_REVISION,
    'model_license': MODEL_LICENSE,
    'runtime': {'python': platform.python_version(), 'torch': torch.__version__, 'transformers': transformers.__version__, 'peft': importlib.metadata.version('peft'), 'device': pipe.device, 'quantized_4bit': pipe.quantized, 'compute_dtype': pipe.compute_dtype},
}
with open('outputs/language_model_artifact_inference_result.json', 'w', encoding='utf-8') as handle:
    json.dump(payload, handle, indent=2, ensure_ascii=False)
with open('outputs/language_model_artifact_inference_generations.csv', 'w', encoding='utf-8', newline='') as handle:
    writer = csv.writer(handle)
    writer.writerow(['prompt', 'kind', 'answer'])
    for row in rows:
        writer.writerow([row['prompt'], 'base-greedy', row['base']])
        writer.writerow([row['prompt'], 'adapted-greedy', row['adapted']])
    for answer in sampled:
        writer.writerow([PROMPTS[0], 'adapted-sampled', answer])
print(json.dumps({k: v for k, v in report.items() if k != 'probes'}, indent=2))
print(sorted(os.listdir('outputs')))

## Interpretation and limits

A successful run proves that an independently supplied adapter bundle is internally consistent, names the carried module's pinned base model at its immutable revision, attaches to the digest-verified base without any network fallback, and changes the model's answers when switched on — without the repository being reachable. It does **not** authenticate the producer, and it establishes no task quality: the evaluation report says `not-measurable` because no labelled prompts exist here, the greedy answers are a reproducibility check rather than a product setting, and the sampled answers are illustrations. Never bypass a failed archive, digest, provenance or base-identity check; obtain a correct bundle from a trusted producer. If base-model acquisition fails, the only acceptable weights are the files named by the inline manifest, never a substitute.

Successful execution proves that the recorded repository revision's pipeline module, carried in this notebook, can acquire and digest-verify the pinned base snapshot, validate and attach an external adapter bundle, validate the supplied prompts, execute the public generation path with the adapter off and on, and emit the shown machine-readable outputs in the tested runtime. It does **not** establish benchmark superiority, deployment calibration, safety for high-consequence decisions, or production fitness on an unseen domain.

**When a check fails:** `Whole-ZIP SHA-256 mismatch` — the archive is not the one whose digest you were given; get it again, do not "fix" the expected hash. `SHA-256 mismatch: <file>` — a file inside the bundle differs from its manifest entry; the bundle was altered or corrupted in transit. `Manifest/file-set mismatch` — files were added or removed. `refusing to attach` — the adapter was trained on a different base model or revision than this pipeline pins; use the matching pipeline. `40-character commit SHA` — the producer recorded a branch name; the bundle is not reproducibly attributable.

**Next experiments:** compare the greedy adapted answer with several sampled ones and note which loops; type a prompt in English and see whether the adapter still answers in Filipino; hand the same prompts and a labelled answer key to your own evaluation to obtain a measurable verdict.

## References

- Repository README: https://github.com/kurtvalcorza/language-model-pipeline/blob/main/README.md
- Repository model card: https://github.com/kurtvalcorza/language-model-pipeline/blob/main/weights/smollm2-360m/MODEL_CARD.md
- Weight provenance: https://github.com/kurtvalcorza/language-model-pipeline/blob/main/weights/README.md
- E2E companion (produces the bundle): https://github.com/kurtvalcorza/language-model-pipeline/blob/main/tutorials/language_model_finetuning_colab.ipynb
- Upstream model: https://huggingface.co/HuggingFaceTB/SmolLM2-360M-Instruct
- Upstream code: https://github.com/huggingface/smollm
- PEFT: https://github.com/huggingface/peft